## Extraction FastF1

In [ ]:

import time
import pandas as pd
import numpy as np
import fastf1
from pathlib import Path
from fastf1.req import RateLimitExceededError

# ================= CONFIG (TON ARCHITECTURE) =================
SEASONS = [2021, 2022, 2023, 2024, 2025]
SESSION_TYPES = ["R", "Q"]  # Race + Qualifying
# Racine = dossier FIL-ROUGE
PROJECT_DIR = Path.cwd().parent

# Dossier cache
CACHE_DIR = PROJECT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Dossier data/raw
DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Activation cache FastF1
fastf1.Cache.enable_cache(str(CACHE_DIR))

OUT_FILE = DATA_RAW_DIR / "fastf1_kpis_R_Q_2021_2025.csv"


# ================= HELPERS =================
def safe_float(series: pd.Series) -> float:
    """Retourne float si existe et non-NaN, sinon NaN."""
    if series is None or len(series) == 0:
        return np.nan
    v = series.values[0]
    return float(v) if pd.notna(v) else np.nan


def extract_session_kpis(season: int, gp_name: str, session_type: str) -> pd.DataFrame:
    """
    Extrait des KPIs par pilote pour une session (R ou Q).
    KPIs : best/avg/std lap times + max speed (fastest lap telemetry) + grid/finish/qualy position.
    """
    session = fastf1.get_session(season, gp_name, session_type)
    session.load()

    # laps (quicklaps = tours représentatifs)
    laps = session.laps.pick_quicklaps()

    # résultats officiels session (contient positions)
    results_df = session.results

    # infos event
    round_no = int(session.event.RoundNumber) if pd.notna(session.event.RoundNumber) else np.nan
    circuit = session.event.Location

    output = []

    # si pas de laps (cas rare), on retourne vide
    if laps.empty:
        return pd.DataFrame()

    for driver in laps["Driver"].unique():
        driver_laps = laps.pick_driver(driver)
        if driver_laps.empty:
            continue

        best_lap = driver_laps["LapTime"].min()
        avg_lap = driver_laps["LapTime"].mean()
        std_lap = driver_laps["LapTime"].std()

        # Telemetry: max speed sur le fastest lap du pilote
        max_speed = np.nan
        try:
            fastest = driver_laps.pick_fastest()
            telemetry = fastest.get_telemetry()
            if telemetry is not None and "Speed" in telemetry.columns:
                max_speed = float(telemetry["Speed"].max())
        except Exception:
            pass

        # Positions (on laisse en float pour supporter NaN)
        grid_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "GridPosition"])
        finish_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"])
        qualy_pos = safe_float(results_df.loc[results_df["Abbreviation"] == driver, "Position"]) if session_type == "Q" else np.nan

        output.append({
            "season": season,
            "round": round_no,
            "gp": gp_name,
            "session": session_type,
            "circuit": circuit,
            "driverCode": driver,

            "bestLapTime_sec": best_lap.total_seconds() if pd.notna(best_lap) else np.nan,
            "avgLapTime_sec": avg_lap.total_seconds() if pd.notna(avg_lap) else np.nan,
            "stdLapTime_sec": std_lap.total_seconds() if pd.notna(std_lap) else np.nan,
            "maxSpeed_kmh": max_speed,

            "gridPosition": grid_pos,
            "finishPosition": finish_pos,
            "qualyPosition": qualy_pos,
        })

    return pd.DataFrame(output)


def load_done_keys(out_file: Path) -> set:
    """Clé d'unité = (season, round, session)."""
    if not out_file.exists():
        return set()
    df = pd.read_csv(out_file)
    # si fichier vide ou colonnes absentes
    if df.empty or not all(c in df.columns for c in ["season", "round", "session"]):
        return set()
    return set(zip(df["season"], df["round"], df["session"]))


# ================= MAIN (INCREMENTAL + RESUME) =================
done = load_done_keys(OUT_FILE)
print(f"✅ Sessions déjà extraites: {len(done)}")

for season in SEASONS:
    schedule = fastf1.get_event_schedule(season)

    # pour éviter une explosion d'appels, on itère proprement
    for _, row in schedule.iterrows():
        gp_name = row["EventName"]
        round_no = int(row["RoundNumber"]) if pd.notna(row["RoundNumber"]) else None

        for session_type in SESSION_TYPES:
            key = (season, round_no, session_type)
            if key in done:
                print(f"↩️ SKIP {season} round {round_no} {session_type} (déjà fait)")
                continue

            tries = 0
            while True:
                tries += 1
                try:
                    df_gp = extract_session_kpis(season, gp_name, session_type=session_type)

                    if df_gp.empty:
                        print(f"⚠️ Vide: {season} round {round_no} {session_type} - {gp_name}")
                        break

                    # append incremental
                    header = not OUT_FILE.exists()
                    df_gp.to_csv(OUT_FILE, mode="a", index=False, header=header, encoding="utf-8")

                    done.add(key)
                    print(f"✔ Saved {season} round {round_no} {session_type} - {gp_name} (rows={len(df_gp)})")

                    # throttle (réduit pression)
                    time.sleep(3.0)
                    break

                except RateLimitExceededError as e:
                    wait_min = 20
                    print(f"⏳ RATE LIMIT: pause {wait_min} min puis reprise... ({e})")
                    time.sleep(wait_min * 60)

                except Exception as e:
                    if tries >= 3:
                        print(f"❌ FAIL {season} round {round_no} {session_type} - {gp_name}: {e}")
                        break
                    print(f"⚠️ RETRY {tries}/3 {season} round {round_no} {session_type} - {gp_name}: {e}")
                    time.sleep(10)

print(f"✅ Terminé. Fichier: {OUT_FILE}")
